In [1]:
# ** This cell is needed since we are not in the src directory 
import sys 
import os
# Add the src/ directory to the Python path
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), "../src/")))

ROOT_DIR = ".."
SRC_DIR = ROOT_DIR + "/src"

import sys
sys.path.append("/Users/admin/eeg-ds004504/")


In [2]:
from config_handler import initiate_config, load_config

initiate_config()

{'data_path': '/Users/admin/eeg-ds004504',
 'derivatives': False,
 'freqBands': {'Alpha': [8, 12],
  'Beta': [12, 30],
  'Delta': [0.5, 4],
  'Theta': [4, 8]},
 'method': 'welch',
 'stepSize': 1.5,
 'windowLength': 3}

In [3]:
print(load_config())

{'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}


In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, ArrayType, MapType
import pandas as pd

In [5]:
# Check if there's an active Spark context and stop it
from pyspark import SparkContext
if SparkContext._active_spark_context:
    print("Stopping existing Spark context...")
    SparkContext._active_spark_context.stop()
    print("Previous Spark context stopped successfully")

In [6]:
# creating the ultimate most optimised spark session ever muwahaha
import os
from pyspark.sql import SparkSession

# Set Java options for the JVM running Spark
# -Xmx12g : Sets the maximum heap size to 12GB
# -Xms4g : Sets the initial heap size to 4GB to avoid resizing overhead
os.environ["_JAVA_OPTIONS"] = "-Xmx12g -Xms4g"

# Build Spark session with memory, parallelism, and network settings
spark = (
    SparkSession.builder 
    # Application name shown in Spark UI
    .appName("EEG_Analysis") 

    # Use all available logical cores or specify a number
    # "local[*]" uses all available cores, "local[12]" limits to 12 threads
    .config("spark.master", "local[12]") \

    # Executor memory: how much memory each Spark worker can use
    .config("spark.executor.memory", "8g") \

    # Driver memory: memory available to the Spark driver (main Python process)
    .config("spark.driver.memory", "8g") \

    # Number of shuffle partitions (e.g., after groupBy, join, etc.)
    # Lower this in local mode to reduce overhead (default is 200)
    .config("spark.sql.shuffle.partitions", "12") \

    # Default number of partitions in operations like parallelize
    .config("spark.default.parallelism", "12") \

    # Maximum size (in MB) allowed for any RPC message (e.g., large UDF closures or data broadcasts)
    .config("spark.rpc.message.maxSize", "256") \

    # Required for avoiding binding issues on some MacOS environments
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .config("spark.driver.host", "127.0.0.1") 
    .getOrCreate()
)

    # ----------------------------------------
    # Additional advanced options (optional):
    # ----------------------------------------

    # Use Kryo serializer instead of default Java serializer for better performance
    # .config("spark.serializer", "org.apache.spark.serializer.KryoSerializer") \

    # Increase broadcast join timeout (in seconds) for large models or lookup tables
    # .config("spark.sql.broadcastTimeout", "600") \

    # Fraction of JVM memory reserved for execution and storage (default is 0.6)
    # .config("spark.memory.fraction", "0.8") \

    # Portion of memory reserved for caching/storage (default is 0.5 of memory.fraction)
    # .config("spark.memory.storageFraction", "0.3") \

    # Enable Apache Arrow for efficient pandas-to-Spark conversion (useful with UDFs)
    # .config("spark.sql.execution.arrow.pyspark.enabled", "true") \

    # Finalize and create the Spark session


spark = SparkSession.builder.appName("MyApp").getOrCreate()

print("New Spark session created successfully")

Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Picked up _JAVA_OPTIONS: -Xmx12g -Xms4g
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/04/10 13:36:28 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/04/10 13:36:29 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


New Spark session created successfully


25/04/10 13:36:29 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


In [7]:
# adding dependencies to spark's context so spark workers (the threads) can access them

In [8]:
def reload_my_modules():
    import importlib
    import populate_schemas
    import feature_extraction
    import schema_definition
    importlib.reload(populate_schemas)
    importlib.reload(feature_extraction)
    importlib.reload(schema_definition)

reload_my_modules()


from populate_schemas import load_subjects_df, extract_features_udtf
from feature_extraction import processEpoch, processSub
from schema_definition import get_feature_schema, get_subject_schema

sc = spark.sparkContext # we pass udf/udtf's (user defind functions and user defined table functions) to spark so it can access them

# Making all necessary modules available to spark
try: 
    # oh btw spark is werid about not finding the config but it always finds it somehow not sure how that works not going to look rn tbh
    # ^ so feature_extraction has extra print statements
    sc.addPyFile(os.path.join(SRC_DIR, "feature_extraction.py"))
    print("Added feature_extraction.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "preprocess_sets.py"))
    print("Added preprocess_sets to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "schema_definition.py"))
    print("Added schema_definition.py to the pyspark context")
    sc.addPyFile(os.path.join(SRC_DIR, "config_handler.py"))
    print("Added config_handler.py to the pyspark context")
except Exception as e:
    print(f"Error adding files to SparkContext: {e}")

Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
Added feature_extraction.py to the pyspark context
Added preprocess_sets to the pyspark context
Added schema_definition.py to the pyspark context
Added config_handler.py to the pyspark context


In [9]:
subject_df = load_subjects_df(spark) #this is the .tsv with the information of all the participants

In [10]:
%%time

# this is the magic of pyspark's distributed system: What would take 50 minutes single threaded takes around 4.5
# I did a similiar optimization with joblib where I would multiprocess this steap and it would take around 7

# What's nice is that we can process whole groups pretty easily :)
group_a_spark_df = (
    subject_df
    .filter(subject_df.Group == "A")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)

group_c_spark_df = (
    subject_df
    .filter(subject_df.Group == "C")
    .groupBy("SubjectID")
    .apply(extract_features_udtf)
)


# Use persist to cache the results in memory as we will probably reference this often, so might as well keep it in ram!
result_group_a = group_a_spark_df.persist()
result_group_c = group_c_spark_df.persist()

# Trigger execution with actions
count_a = result_group_a.count()
count_c = result_group_c.count()

print(f"Processed {count_a} records for Alzheimer's group")
print(f"Processed {count_c} records for Control group")

/Users/admin/neuro-venv/lib/python3.9/site-packages/pyspark/sql/pandas/group_ops.py:104: UserWarning: It is preferred to use 'applyInPandas' over this API. This API will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(
Config not found in feature_extraction.py                         (0 + 12) / 12]
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
processSub sub-004
Config not found in feature_extraction.py
Config using in feature Extraction.py {'data_path': '/Users/admin/eeg-ds004504', 'derivatives': False, 'freqBands': {'Alpha': [8, 12], 'Beta': [12, 30], 'Delta': [0.5, 4], 'Theta': [4, 8]}, 'method': 'welch', 'stepSize': 1.5, 'windowLength': 3}
processSub sub-022
subPath sub-004
subPath data_path /Users/admin/eeg-ds004504
Path handed: /Users/admin/eeg-

Processed 2462796 records for Alzheimer's group
Processed 2047122 records for Control group
CPU times: user 147 ms, sys: 182 ms, total: 329 ms
Wall time: 17min 11s


In [11]:
#just renaming things now that we understand the types and where things are coming from
alz_df = group_a_spark_df
cntrl_df = group_c_spark_df

In [13]:
# should to pandas then to pickle
alz_df_pandas = alz_df.toPandas()
cntrl_df_pandas = cntrl_df.toPandas()


In [15]:
alz_df_pandas.to_pickle("alz_df_apr10_1355.pkl")
cntrl_df_pandas.to_pickle("cntrl_df_apr10_1355.pkl")

In [16]:
alz_df.show()

+---------+-------+---------+--------+-----------+-------------+----------+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName| FeatureValue|table_type|
+---------+-------+---------+--------+-----------+-------------+----------+
|  sub-008|   ep-0|      Fp1|   Alpha|      Power|  7.020328E-4|      band|
|  sub-008|   ep-0|      Fp1|    Beta|      Power|  3.399499E-4|      band|
|  sub-008|   ep-0|      Fp1|   Delta|      Power|  0.087231696|      band|
|  sub-008|   ep-0|      Fp1|   Theta|      Power| 0.0011391409|      band|
|  sub-008|   ep-0|      Fp1|    NULL|TotalEnergy|   0.34805238| electrode|
|  sub-008|   ep-0|      Fp1|    NULL| TotalPower|  0.011235955| electrode|
|  sub-008|   ep-0|      Fp2|   Alpha|      Power| 0.0014689578|      band|
|  sub-008|   ep-0|      Fp2|    Beta|      Power|  5.043481E-4|      band|
|  sub-008|   ep-0|      Fp2|   Delta|      Power|   0.08462298|      band|
|  sub-008|   ep-0|      Fp2|   Theta|      Power|  0.002023743|      band|
|  sub-008| 

# The start of data processing

In [282]:
# give each its respective labels
from pyspark.sql.functions import lit
alz_df = alz_df.withColumn("label", lit(1)).repartition(16).persist()
cntrl_df = cntrl_df.withColumn("label", lit(0)).repartition(16).persist()

In [283]:
# union everything
full_df = alz_df.unionByName(cntrl_df)


In [284]:
from pyspark.sql.functions import col

# Split based on feature type
band_df = full_df.filter(col("table_type") == "band")
channel_df = full_df.filter(col("table_type") == "electrode")
epoch_df = full_df.filter(col("table_type") == "epoch")


In [285]:
band_df.show(3)
channel_df.show(3)
epoch_df.show(3)

+---------+-------+---------+--------+-----------+------------+----------+-----+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|label|
+---------+-------+---------+--------+-----------+------------+----------+-----+
|  sub-034| ep-515|       Fz|   Delta|      Power|  0.08544011|      band|    1|
|  sub-019| ep-508|       T4|   Delta|      Power|  0.06714194|      band|    1|
|  sub-011|  ep-66|       T3|   Alpha|      Power|0.0036147116|      band|    1|
+---------+-------+---------+--------+-----------+------------+----------+-----+
only showing top 3 rows

+---------+-------+---------+--------+-----------+------------+----------+-----+
|SubjectID|EpochID|Electrode|WaveBand|FeatureName|FeatureValue|table_type|label|
+---------+-------+---------+--------+-----------+------------+----------+-----+
|  sub-025| ep-356|       T5|    NULL|TotalEnergy|  0.29269218| electrode|    1|
|  sub-019| ep-319|       F8|    NULL| TotalPower| 0.011235955| electrode|    1|
|  

In [286]:
from pyspark.sql.functions import concat_ws

# Band-level: Electrode_WaveBand_Feature
band_df = band_df.withColumn("pivot", concat_ws("_", "Electrode", "WaveBand", "FeatureName"))

# Channel-level: Electrode_Feature
channel_df = channel_df.withColumn("pivot", concat_ws("_", "Electrode", "FeatureName"))

# Epoch-level: just FeatureName
epoch_df = epoch_df.withColumn("pivot", col("FeatureName"))


In [287]:
from pyspark.sql.functions import first

# Pivot band-level features
band_pivot = band_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot channel-level features
channel_pivot = channel_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))

# Pivot epoch-level features
epoch_pivot = epoch_df.groupBy("SubjectID", "EpochID", "label").pivot("pivot").agg(first("FeatureValue"))


In [288]:
print("*", end ="")
print(band_pivot.head(1))
print("\n*", end="")
print(channel_pivot.head(1))
print("\n*", end="")
print(epoch_pivot.head(1))

*

[Row(SubjectID='sub-001', EpochID='ep-308', label=1, C3_Alpha_Power=0.0010492837755009532, C3_Beta_Power=0.00025931705022230744, C3_Delta_Power=0.08566871285438538, C3_Theta_Power=0.0025874667335301638, C4_Alpha_Power=0.0007275737007148564, C4_Beta_Power=0.00027581985341385007, C4_Delta_Power=0.08677294850349426, C4_Theta_Power=0.0018226978136226535, Cz_Alpha_Power=0.0008660402963869274, Cz_Beta_Power=0.000240262525039725, Cz_Delta_Power=0.0865846648812294, Cz_Theta_Power=0.002016837475821376, F3_Alpha_Power=0.0006777072558179498, F3_Beta_Power=0.0002923805150203407, F3_Delta_Power=0.08563768863677979, F3_Theta_Power=0.0028387021739035845, F4_Alpha_Power=0.000987324514426291, F4_Beta_Power=0.0002887095615733415, F4_Delta_Power=0.08617019653320312, F4_Theta_Power=0.0020574666559696198, F7_Alpha_Power=0.0006861757137812674, F7_Beta_Power=0.0003673869068734348, F7_Delta_Power=0.08464381843805313, F7_Theta_Power=0.0034037528093904257, F8_Alpha_Power=0.001092611812055111, F8_Beta_Power=0.00

[Row(SubjectID='sub-018', EpochID='ep-417', label=1, C3_TotalEnergy=0.2619585692882538, C3_TotalPower=0.01123595517128706, C4_TotalEnergy=0.28047388792037964, C4_TotalPower=0.01123595517128706, Cz_TotalEnergy=0.2694971561431885, Cz_TotalPower=0.01123595517128706, F3_TotalEnergy=0.275530606508255, F3_TotalPower=0.01123595517128706, F4_TotalEnergy=0.322449654340744, F4_TotalPower=0.01123595517128706, F7_TotalEnergy=0.2650219798088074, F7_TotalPower=0.01123595517128706, F8_TotalEnergy=0.308745801448822, F8_TotalPower=0.01123595517128706, Fp1_TotalEnergy=0.3290731906890869, Fp1_TotalPower=0.01123595517128706, Fp2_TotalEnergy=0.328632652759552, Fp2_TotalPower=0.01123595517128706, Fz_TotalEnergy=0.294182151556015, Fz_TotalPower=0.01123595517128706, O1_TotalEnergy=0.2754564583301544, O1_TotalPower=0.01123595517128706, O2_TotalEnergy=0.27404552698135376, O2_TotalPower=0.01123595517128706, P3_TotalEnergy=0.2251250296831131, P3_TotalPower=0.01123595517128706, P4_TotalEnergy=0.2387922704219818, P

[Stage 151475:========================================>           (25 + 7) / 32]

[Row(SubjectID='sub-019', EpochID='ep-199', label=1, AppEntropy=1.098935842514038, HiguchiFD=1.921994924545288, HjorthComplexity=3.0158233642578125, HjorthMobility=0.38199466466903687, KatzFD=2.8683042526245117, Kurtosis=3.6828529834747314, Mean=1.2332661934751835e-20, RMS=2.7313768441672437e-05, SampleEntropy=1.0600122213363647, Skewness=-0.5153109431266785, Std=2.732287066464778e-05, Variance=7.490839615265088e-10)]


In [289]:
# !! USING BAND PIVOT ONLY!

In [290]:
from functools import reduce

# full_df = reduce(
#     lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
#     [band_pivot, channel_pivot, epoch_pivot]
# ).fillna(0.0)


full_df = reduce(
    lambda df1, df2: df1.join(df2, on=["SubjectID", "EpochID", "label"], how="outer"),
     [band_pivot, channel_pivot, epoch_pivot]
).fillna(0.0)

In [291]:
NUM_TEST_SUBJECTS_PER_GROUP = 2

# Get test subject IDs from full_df (which has .label)
alz_test_subjects = (
    full_df.filter("label == 1")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

cntrl_test_subjects = (
    full_df.filter("label == 0")
    .select("SubjectID")
    .distinct()
    .orderBy("SubjectID")
    .limit(NUM_TEST_SUBJECTS_PER_GROUP)
    .rdd.flatMap(lambda row: row)
    .collect()
)

test_subjects = alz_test_subjects + cntrl_test_subjects #this will be the firs 2 subjects of each group for reproduceablility


                                                                                2]]

In [292]:
test_subjects

['sub-001', 'sub-002', 'sub-037', 'sub-038']

In [293]:
# Split into test and train sets
test_df = full_df.filter(col("SubjectID").isin(test_subjects))
train_df = full_df.filter(~col("SubjectID").isin(test_subjects))


In [294]:
import dimensionality_reduction
import importlib
importlib.reload(dimensionality_reduction)
from dimensionality_reduction import min_max_normalize
feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]
train_norm_df, test_norm_df = min_max_normalize(train_df, test_df, feature_cols)

                                                                                ]]]

In [295]:
from dimensionality_reduction import normalize_by_column
# feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]
# # we are z-scoring after min max normalizing
# train_norm_df, test_norm_df = normalize_by_column(train_norm_df, test_norm_df, feature_cols)

In [296]:
# import dimensionality_reduction
# import importlib
# importlib.reload(dimensionality_reduction)

# # IMOPORTANT BUT KEPT OUT !


# from dimensionality_reduction import normalize_by_column
# feature_cols = [c for c in train_df.columns if c not in ("SubjectID", "EpochID", "label")]

# train_norm_df, test_norm_df = normalize_by_column(train_df, test_df, feature_cols)


In [297]:
# train_norm_df = train_norm_df.persist()
# test_norm_df = test_norm_df.persist()


In [298]:
train_norm

DataFrame[SubjectID: string, EpochID: string, label: int, C3_TotalEnergy: double, C3_TotalPower: double, C4_TotalEnergy: double, C4_TotalPower: double, Cz_TotalEnergy: double, Cz_TotalPower: double, F3_TotalEnergy: double, F3_TotalPower: double, F4_TotalEnergy: double, F4_TotalPower: double, F7_TotalEnergy: double, F7_TotalPower: double, F8_TotalEnergy: double, F8_TotalPower: double, Fp1_TotalEnergy: double, Fp1_TotalPower: double, Fp2_TotalEnergy: double, Fp2_TotalPower: double, Fz_TotalEnergy: double, Fz_TotalPower: double, O1_TotalEnergy: double, O1_TotalPower: double, O2_TotalEnergy: double, O2_TotalPower: double, P3_TotalEnergy: double, P3_TotalPower: double, P4_TotalEnergy: double, P4_TotalPower: double, Pz_TotalEnergy: double, Pz_TotalPower: double, T3_TotalEnergy: double, T3_TotalPower: double, T4_TotalEnergy: double, T4_TotalPower: double, T5_TotalEnergy: double, T5_TotalPower: double, T6_TotalEnergy: double, T6_TotalPower: double]

In [299]:
pca_input_cols = feature_cols
from dimensionality_reduction import fit_pca_model

pca_model, k_val = fit_pca_model(train_norm_df, pca_input_cols, variance_target=0.95)

print(f"PCA model fitted with {k_val} components to capture 95% variance")


                                                                                12]

PCA model fitted with 20 components to capture 95% variance


In [300]:
from dimensionality_reduction import apply_pca_model

train_df = apply_pca_model(train_norm_df, pca_input_cols, pca_model, k_val)
test_df = apply_pca_model(test_norm_df, pca_input_cols, pca_model, k_val)


# ML

In [301]:
model_summaries = []

In [312]:
%%time
from pyspark.ml.classification import MultilayerPerceptronClassifier

# Get input size from PCA features

mlp = MultilayerPerceptronClassifier(
    featuresCol="features",
    labelCol="label",
    layers=[k_val, 500, 2],  # input → hidden (100 units) → 2 output classes
    maxIter=1000,
    seed=42
)

mlp_model = mlp.fit(train_df)
mlp_preds = mlp_model.transform(test_df)

[Stage 206584:=============================================>        (5 + 1) / 6]2]]

CPU times: user 47.6 ms, sys: 86.3 ms, total: 134 ms
Wall time: 4min 36s


In [313]:
from pyspark.ml.evaluation import BinaryClassificationEvaluator

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
mlp_auc = evaluator.evaluate(mlp_preds)

print("MLP AUC:", mlp_auc)

[Stage 206606:>           (0 + 12) / 12][Stage 206608:>             (0 + 0) / 6]2]

MLP AUC: 0.7661714952362435


In [314]:
preds_pd = mlp_preds.select("prediction", "label").toPandas()

from sklearn.metrics import classification_report, accuracy_score

print("Neural Network accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

[Stage 206719:====>                                               (1 + 11) / 12]2]

Neural Network accuracy: 0.6995581737849779
              precision    recall  f1-score   support

     Control       0.71      0.77      0.74      1112
 Alzheimer's       0.69      0.62      0.65       925

    accuracy                           0.70      2037
   macro avg       0.70      0.69      0.69      2037
weighted avg       0.70      0.70      0.70      2037



In [318]:
from sklearn.metrics import classification_report

report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

model_summaries.append({
    "model": "Neural Network",
    "auc": mlp_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [320]:
from pyspark.ml.classification import GBTClassifier

gbt = GBTClassifier(featuresCol="features", labelCol="label", maxIter=250)
gbt_model = gbt.fit(train_df)
gbt_preds = gbt_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
gbt_auc = evaluator.evaluate(gbt_preds)
print("Gradient Boosted Trees AUC:", gbt_auc)

preds_pd = gbt_preds.select("prediction", "label").toPandas()
print("GBT accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))

ConnectionRefusedError: [Errno 61] Connection refused

In [ ]:
# Generate classification report for GBT
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append GBT results to summary
model_summaries.append({
    "model": "Gradient Boosted Trees",
    "auc": gbt_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [310]:
from pyspark.ml.classification import DecisionTreeClassifier

tree = DecisionTreeClassifier(featuresCol="features", labelCol="label", maxDepth=5)
tree_model = tree.fit(train_df)
tree_preds = tree_model.transform(test_df)

evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
tree_auc = evaluator.evaluate(tree_preds)
print("Decision Tree AUC:", tree_auc)

preds_pd = tree_preds.select("prediction", "label").toPandas()
print("Decision Tree accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


                                                                                12]

Decision Tree AUC: 0.5750097219521679


[Stage 184026:>                                                     (0 + 6) / 6]

Decision Tree accuracy: 0.616593028964163
              precision    recall  f1-score   support

     Control       0.68      0.56      0.61      1112
 Alzheimer's       0.56      0.69      0.62       925

    accuracy                           0.62      2037
   macro avg       0.62      0.62      0.62      2037
weighted avg       0.63      0.62      0.62      2037



In [249]:
# Generate classification report for Decision Tree
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append Decision Tree results to summary
model_summaries.append({
    "model": "Decision Tree",
    "auc": tree_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [250]:
from pyspark.ml.classification import LinearSVC

# Train SVM model
svm = LinearSVC(featuresCol="features", labelCol="label", maxIter=500, regParam=0.1)
svm_model = svm.fit(train_df)
svm_preds = svm_model.transform(test_df)

# Evaluate SVM
from pyspark.ml.evaluation import BinaryClassificationEvaluator
evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName="areaUnderROC")
svm_auc = evaluator.evaluate(svm_preds)
print("SVM AUC:", svm_auc)

# Accuracy and report
preds_pd = svm_preds.select("prediction", "label").toPandas()
from sklearn.metrics import classification_report, accuracy_score

print("SVM accuracy:", accuracy_score(preds_pd["label"], preds_pd["prediction"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names))


SVM AUC: 0.6355940112774644
SVM accuracy: 0.4693176239567992
              precision    recall  f1-score   support

     Control       0.83      0.04      0.07      1112
 Alzheimer's       0.46      0.99      0.63       925

    accuracy                           0.47      2037
   macro avg       0.65      0.51      0.35      2037
weighted avg       0.66      0.47      0.32      2037



In [251]:
# Generate classification report for SVM
report_str = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["label"], preds_pd["prediction"], target_names=target_names, output_dict=True)

# Append SVM results to summary
model_summaries.append({
    "model": "SVM",
    "auc": svm_auc,
    "accuracy": accuracy_score(preds_pd["label"], preds_pd["prediction"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [252]:
from pyspark.ml.feature import BucketedRandomProjectionLSH
from pyspark.ml.linalg import Vectors
from pyspark.sql.functions import col

# Step 1: Fit LSH model on training data
lsh = BucketedRandomProjectionLSH(
    inputCol="features",
    outputCol="hashes",
    bucketLength=0.25,
    numHashTables=6
)
lsh_model = lsh.fit(train_df)

# Step 2: Perform approximate similarity join between test and train
# This will find the approximate nearest neighbors of test samples in train set
similarities = lsh_model.approxSimilarityJoin(
    datasetA=test_df,
    datasetB=train_df,
    threshold=float("inf"),  # You can limit this if needed
    distCol="euclidean_distance"
)

# Step 3: For each test point, pick nearest neighbor (smallest distance)
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

window = Window.partitionBy("datasetA").orderBy("euclidean_distance")

nearest_neighbors = similarities \
    .withColumn("rank", row_number().over(window)) \
    .filter(col("rank") == 1)

# Step 4: Collect prediction from nearest training label
predictions = nearest_neighbors.select(
    col("datasetA.label").alias("true_label"),
    col("datasetB.label").alias("predicted_label")
)

# Step 5: Evaluate
preds_pd = predictions.toPandas()

from sklearn.metrics import accuracy_score, classification_report

print("Approximate KNN accuracy:", accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]))

target_names = ["Control", "Alzheimer's"]
print(classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names))


ERROR:root:KeyboardInterrupt while sending command.               (0 + 12) / 12]
Traceback (most recent call last):
  File "/Users/admin/neuro-venv/lib/python3.9/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
  File "/Users/admin/neuro-venv/lib/python3.9/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
  File "/Library/Developer/CommandLineTools/Library/Frameworks/Python3.framework/Versions/3.9/lib/python3.9/socket.py", line 704, in readinto
    return self._sock.recv_into(b)
KeyboardInterrupt


KeyboardInterrupt: 

In [ ]:
# Ensure columns are correctly named
if "true_label" not in preds_pd.columns:
    preds_pd.columns = ["true_label", "predicted_label"]

# Generate classification report
report_str = classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names, output_dict=False)
report_dict = classification_report(preds_pd["true_label"], preds_pd["predicted_label"], target_names=target_names, output_dict=True)

# Append Approximate KNN results to summary
model_summaries.append({
    "model": "Approximate KNN",
    "auc": "N/A",
    "accuracy": accuracy_score(preds_pd["true_label"], preds_pd["predicted_label"]),
    "report_str": report_str,
    "report_dict": report_dict
})


In [ ]:
print("\n\n=== MODEL PERFORMANCE SUMMARY ===")
for summary in model_summaries:
    print(f"\nModel: {summary['model']}")
    
    auc = summary.get('auc', 'N/A')
    if isinstance(auc, (int, float)):
        print(f"  AUC:      {auc:.4f}")
    else:
        print(f"  AUC:      {auc}")

    acc = summary.get('accuracy', 'N/A')
    if isinstance(acc, (int, float)):
        print(f"  Accuracy: {acc:.4f}")
    else:
        print(f"  Accuracy: {acc}")
    
    print("\n  Classification Report:")
    print(summary.get("report_str", "  No report available"))

print("\n\n=== ACTIVE CONFIGURATION ===")
from pprint import pprint
pprint(load_config())
from datetime import datetime

now = datetime.now()
print("Current date and time:", now)

In [ ]:

import sys
from datetime import datetime

# Open the log file in append mode
logfile = open("model_results_log.txt", "a")

# Define a dual-output print function
def log_print(*args, **kwargs):
    print(*args, **kwargs)
    print(*args, **kwargs, file=logfile)

# === Your original code, now using `log_print` ===
log_print("\n\n=== MODEL PERFORMANCE SUMMARY ===")
for summary in model_summaries:
    log_print(f"\nModel: {summary['model']}")
    
    auc = summary.get('auc', 'N/A')
    if isinstance(auc, (int, float)):
        log_print(f"  AUC:      {auc:.4f}")
    else:
        log_print(f"  AUC:      {auc}")

    acc = summary.get('accuracy', 'N/A')
    if isinstance(acc, (int, float)):
        log_print(f"  Accuracy: {acc:.4f}")
    else:
        log_print(f"  Accuracy: {acc}")
    
    log_print("\n  Classification Report:")
    log_print(summary.get("report_str", "  No report available"))

log_print("\n\n=== ACTIVE CONFIGURATION ===")
from pprint import pprint
from io import StringIO

# Capture pprint output
config_str = StringIO()
pprint(load_config(), stream=config_str)
log_print(config_str.getvalue())

log_print("features created")
pca_input_cols.sort()
log_print(pca_input_cols)


log_print("Normalized with -1 to 1")
# Add date and time
now = datetime.now()
log_print("Current date and time:", now)

logfile.close()  # Always close the file when done
